# Titanic Survival — Preprocessing Pipeline and Cleaned Dataset
## Project 2

**Goal:** Build a reproducible, leakage-safe preprocessing pipeline and export a clean,
machine-learning-ready Titanic dataset.

The transformer is fitted **only on the training split**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

BASE_DIR=Path.cwd()
if not (BASE_DIR/"dataset").exists(): BASE_DIR=BASE_DIR.parent
DATA_PATH=BASE_DIR/"dataset"/"titanic_train.csv"
OUT=BASE_DIR/"outputs"
OUT.mkdir(exist_ok=True)
df=pd.read_csv(DATA_PATH)
display(df.head())
print("Shape:",df.shape)

## 1. Missing-Value Inspection

In [ ]:
missing = pd.DataFrame({
    "column":df.columns,
    "missing_count":df.isna().sum().values,
    "missing_percentage":(df.isna().mean()*100).round(2).values
})
display(missing)
print("Total missing cells:",int(df.isna().sum().sum()))

## 2. Feature Engineering

In [ ]:
work=df.copy()
work["FamilySize"]=work["SibSp"]+work["Parch"]+1
work["IsAlone"]=(work["FamilySize"]==1).astype(int)
work["FarePerPerson"]=work["Fare"]/work["FamilySize"]
work["Title"]=work["Name"].str.extract(r",\s*([^.]*)\.",expand=False).str.strip()
common_titles=["Mr","Miss","Mrs","Master"]
work["Title"]=work["Title"].where(work["Title"].isin(common_titles),"Rare")
display(work[["Name","FamilySize","IsAlone","FarePerPerson","Title"]].head(10))

## 3. Human-Readable Cleaning

In [ ]:
clean=work.copy()
clean["Age"]=clean["Age"].fillna(clean["Age"].median())
clean["Embarked"]=clean["Embarked"].fillna(clean["Embarked"].mode()[0])
clean["Fare"]=clean["Fare"].fillna(clean["Fare"].median())
clean["FarePerPerson"]=clean["FarePerPerson"].replace([np.inf,-np.inf],np.nan)
clean["FarePerPerson"]=clean["FarePerPerson"].fillna(clean["FarePerPerson"].median())
clean.to_csv(OUT/"titanic_cleaned_readable.csv",index=False)
print("Missing cells after cleaning:",int(clean.isna().sum().sum()))
display(clean.head())

## 4. Train/Test Split

In [ ]:
features=["Pclass","Sex","Age","SibSp","Parch","Fare","Embarked",
          "FamilySize","IsAlone","FarePerPerson","Title"]
X=work[features].copy()
y=df["Survived"].copy()

X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.20,random_state=42,stratify=y
)
print(X_train.shape,X_test.shape)

## 5. Define Preprocessing Pipelines

In [ ]:
numeric=["Pclass","Age","SibSp","Parch","Fare","FamilySize","IsAlone","FarePerPerson"]
categorical=["Sex","Embarked","Title"]

numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

categorical_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore",sparse_output=False))
])

preprocessor=ColumnTransformer([
    ("numeric",numeric_pipeline,numeric),
    ("categorical",categorical_pipeline,categorical)
])
display(preprocessor)

## 6. Fit on Training Data Only

In [ ]:
# Critical leakage-prevention step.
X_train_p=preprocessor.fit_transform(X_train)
X_test_p=preprocessor.transform(X_test)

print("Processed train:",X_train_p.shape)
print("Processed test:",X_test_p.shape)

## 7. Export Processed Datasets

In [ ]:
names=preprocessor.get_feature_names_out()

train=pd.DataFrame(X_train_p,columns=names,index=X_train.index)
train["Survived"]=y_train

test=pd.DataFrame(X_test_p,columns=names,index=X_test.index)
test["Survived"]=y_test

train.to_csv(OUT/"train_preprocessed_with_target.csv",index=False)
test.to_csv(OUT/"test_preprocessed_with_target.csv",index=False)
train.drop(columns="Survived").to_csv(OUT/"X_train_preprocessed.csv",index=False)
test.drop(columns="Survived").to_csv(OUT/"X_test_preprocessed.csv",index=False)

pd.DataFrame({"processed_feature":names}).to_csv(
    OUT/"processed_feature_map.csv",index=False
)
print("Export complete.")

## 8. Before vs After Missing Values

In [ ]:
print("Before:",int(df.isna().sum().sum()))
print("After :",int(clean.isna().sum().sum()))

plt.figure(figsize=(8,5))
plt.bar(["Before","After"],[df.isna().sum().sum(),clean.isna().sum().sum()])
plt.title("Missing Cells Before vs After Cleaning")
plt.ylabel("Missing Cells")
plt.show()

## 9. Leakage-Safety Explanation

- Split raw data into train and test first.
- Median imputation learns statistics from training data only.
- Most-frequent categorical imputation learns from training data only.
- StandardScaler learns its parameters from training data only.
- OneHotEncoder is fitted on training categories and ignores unknown test categories.
- The fitted transformer is reused for the test set.

Therefore, test-set information is not used to learn preprocessing parameters.

## 10. Conclusion

The Titanic dataset is now available in both human-readable cleaned form and encoded/scaled
machine-learning-ready form. The preprocessing workflow is reproducible and designed to avoid
data leakage, making the exported training data suitable for the next modeling project.